# Неделя 5. Форматы хранения на примере книжного магазина

Сначала загрузим каталог поставщика, затем проверим, как меняется схема каталога. На истории заказов посмотрим, какую часть данных нужно прочитать для запроса и сколько места занимают разные представления.

| Аспект лекции | Что делает [исходный семинар](sem.ipynb) | Что проверяем здесь |
| --- | --- | --- |
| [Текстовые форматы, слайды 13–18](../week05_file_formats.pdf#page=13) | Читает XML в Spark и ищет максимальную цену | Сравниваем два `rowTag` и считаем максимум для строк и чисел |
| [Эволюция схемы и Avro, слайды 19–25](../week05_file_formats.pdf#page=19) | Показывает схему из файла, добавление поля и конфликт типов | Читаем старые и новые файлы обеими схемами и проверяем роль `default` |
| [Колоночное хранение, слайды 27–32](../week05_file_formats.pdf#page=27) | Записывает небольшой Parquet-файл | Измеряем размер выбранных колонок по футеру и смотрим, какие row groups можно пропустить |
| [Partition discovery, слайды 36–41](../week05_file_formats.pdf#page=36) | Записывает Parquet с разбивкой по городу | Находим партиции по путям файлов и сравниваем набор файлов до и после фильтра |
| [Кодирование, сжатие и сортировка, слайды 29–30](../week05_file_formats.pdf#page=29) | Сравнивает размеры CSV, JSON, Avro и Parquet на данных Kaggle | Пишем одни и те же записи, отдельно меняем кодек Parquet и порядок строк |

## Запуск

Используйте ядро Jupyter с Python 3.10–3.12. Для XML-демо нужен Spark 4.0 и JDK 17 или 21. При необходимости установите библиотеки в окружение ядра, затем перезапустите его.

```python
%pip install "pyspark==4.0.1" "avro==1.12.0" "pyarrow==21.0.0"
```

Дальше можно выполнить ячейки сверху вниз. Если Java недоступна, выполните подготовку и перейдите к разделу 2. Все входные данные создаются в ноутбуке. Файлы опытов лежат во временной папке ядра, новый запуск подготовки создаёт новую папку.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from IPython.display import Markdown, SVG, display

workdir = TemporaryDirectory(prefix="week05-formats-")
root = Path(workdir.name).resolve()


def show_table(headers, rows):
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    lines.extend("| " + " | ".join(map(str, row)) + " |" for row in rows)
    display(Markdown("\n".join(lines)))

## 1. XML в Spark SQL. Где начинается строка и откуда берётся тип?

**Предскажите результат.** Сколько строк получится с `rowTag=Catalog` и с `rowTag=Book`? Какая строка больше при сравнении как текст — `44.95` или `9.50`?

В каталоге две книги. У первой два тега, а у обеих SKU с ведущими нулями. Сначала посмотрим, как Spark выводит схему. Затем прочитаем цену как текст и посчитаем оба максимума в одном SQL-запросе.

In [ ]:
books_xml = root / "books.xml"
books_xml.write_text(
    '''<?xml version="1.0" encoding="UTF-8"?>
<Catalog>
  <Book id="bk101">
    <Title>XML in Practice</Title>
    <SKU>00123</SKU>
    <Price>44.95</Price>
    <Tag>xml</Tag>
    <Tag>spark</Tag>
  </Book>
  <Book id="bk102">
    <Title>Data Formats</Title>
    <SKU>00124</SKU>
    <Price>9.50</Price>
    <Tag>storage</Tag>
  </Book>
</Catalog>
''',
    encoding="utf-8",
)
print(books_xml.read_text(encoding="utf-8"))

In [ ]:
from decimal import Decimal

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master("local[2]")
    .appName("week05-file-formats")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.warehouse.dir", (root / "spark-warehouse").as_uri())
    .getOrCreate()
)
try:
    catalog = spark.read.xml(books_xml.as_uri(), rowTag="Catalog")
    books = spark.read.xml(books_xml.as_uri(), rowTag="Book")
    row_counts = [catalog.count(), books.count()]
    show_table(
        ["rowTag", "Строк в DataFrame", "Колонки верхнего уровня"],
        [("Catalog", row_counts[0], ", ".join(catalog.columns)),
         ("Book", row_counts[1], ", ".join(books.columns))],
    )
    assert row_counts == [1, 2]
    books.printSchema()
    books.orderBy("_id").show(truncate=False)

    spark.sql(f'''
        CREATE OR REPLACE TEMP VIEW books_text
        USING xml
        OPTIONS (path '{books_xml.as_uri()}', rowTag 'Book', inferSchema 'false')
    ''')
    spark.sql("SELECT _id, SKU, Price FROM books_text ORDER BY _id").show()
    maxima = spark.sql('''
        SELECT MAX(Price) AS max_text,
               MAX(CAST(Price AS DECIMAL(8, 2))) AS max_decimal
        FROM books_text
    ''')
    maxima.show()
    result = maxima.first()
    assert (result.max_text, result.max_decimal) == ("9.50", Decimal("44.95"))
finally:
    spark.stop()

`Catalog` даёт одну строку с вложенным массивом книг, `Book` — по строке на книгу. Атрибут `id` попадает в `_id`, а повторяющийся `Tag` становится массивом. При выводе схемы Spark превращает числовой на вид SKU в число, а при чтении как текста остаётся `00123`. В рабочем контракте стоит задать SKU строкой, а цену десятичным числом.

## 2. Avro. Переживёт ли читатель изменение схемы?

У каталога v1 есть идентификатор книги и цена в копейках. В этой версии валюта подразумевается. В v2 появляется `currency`. Для старых записей новый читатель подставляет `RUB`. Новая запись использует `EUR`, чтобы в результате было видно, когда значение пришло из файла.

Avro хранит **схему записи** в заголовке файла. Приложение передаёт **схему чтения** отдельно.

**Предскажите результат.** Какие из четырёх сочетаний «файл v1/v2 × читатель v1/v2» сработают? Что изменится, если убрать `default` из схемы нового читателя или записать цену строкой?

In [ ]:
import json

import avro.schema
from avro.datafile import DataFileReader, DataFileWriter
from avro.errors import SchemaResolutionException
from avro.io import DatumReader, DatumWriter


def write_avro(path, schema, records):
    with DataFileWriter(path.open("wb"), DatumWriter(), schema, codec="null") as writer:
        for record in records:
            writer.append(record)


def read_avro(path, reader_schema=None):
    with DataFileReader(path.open("rb"), DatumReader(readers_schema=reader_schema)) as reader:
        writer_schema = json.loads(reader.meta["avro.schema"])
        return writer_schema, list(reader)


book_v1_fields = [
    {"name": "book_id", "type": "string"},
    {"name": "price_cents", "type": "int"},
]
currency_field = {"name": "currency", "type": "string"}


def book_schema(fields):
    return avro.schema.parse(json.dumps({"type": "record", "name": "Book", "fields": fields}))


book_v1 = book_schema(book_v1_fields)
default_currency = "RUB"
book_v2 = book_schema([*book_v1_fields, {**currency_field, "default": default_currency}])
book_v2_required = book_schema([*book_v1_fields, currency_field])
book_price_text = book_schema([
    book_v1_fields[0],
    {"name": "price_cents", "type": "string"},
])

old_book = {"book_id": "bk101", "price_cents": 4495}
new_book = {**old_book, "currency": "EUR"}
v1_path = root / "v1.avro"
v2_path = root / "v2.avro"
text_price_path = root / "text-price.avro"
write_avro(v1_path, book_v1, [old_book])
write_avro(v2_path, book_v2, [new_book])
write_avro(text_price_path, book_price_text, [{**old_book, "price_cents": "4495"}])

stored_schema, _ = read_avro(v2_path)
print("Схема записи из заголовка v2.avro:")
print(json.dumps(stored_schema, ensure_ascii=False, indent=2))

In [ ]:
avro_cases = [
    ("v1", v1_path, "v1", book_v1, old_book),
    ("v1", v1_path, "v2", book_v2, {**old_book, "currency": default_currency}),
    ("v2", v2_path, "v1", book_v1, old_book),
    ("v2", v2_path, "v2", book_v2, new_book),
    ("v1", v1_path, "v2 без default", book_v2_required, None),
    ("цена как string", text_price_path, "v1", book_v1, None),
]
compatibility_rows = []
for writer_version, path, reader_version, schema, expected in avro_cases:
    try:
        _, records = read_avro(path, schema)
    except SchemaResolutionException as error:
        assert expected is None
        outcome = f"Ошибка: {str(error).splitlines()[0]}"
    else:
        assert records == [expected]
        outcome = f"`{records[0]}`"
    compatibility_rows.append((writer_version, reader_version, outcome))

show_table(["Файл записан схемой", "Схема читателя", "Результат"], compatibility_rows)

- При **обратной совместимости** читатель v2 читает файл v1 и подставляет `RUB`. Без `default` чтение падает.
- При **прямой совместимости** читатель v1 читает файл v2 и пропускает незнакомое поле `currency`.
- Запись цены строкой ломает чтение по схеме с `int`. Это рабочий Avro-файл со схемой `string`, поэтому проблема проявляется при выборе схемы читателя. [Правила согласования схем Avro](https://avro.apache.org/docs/1.12.0/specification/#schema-resolution)

## 3. Parquet. Какие части файла нужны запросу?

Сгенерируем 12 000 заказов. В каждом файле будет шесть **row groups** по 2 000 строк. Внутри каждой группы данные каждой колонки образуют свой **column chunk**. Сохраним одни и те же записи в двух порядках — по `order_id` и после перемешивания с фиксированным seed.

In [ ]:
from random import Random

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq

n_orders = 12_000
rows_per_group = 2_000
markets = ("ru", "eu", "apac")
channels = ("web", "mobile", "store")

orders = [
    {
        "order_id": i,
        "book_id": f"bk{101 + i % 120:03d}",
        "market": markets[(i - 1) % len(markets)],
        "channel": channels[(i // 7) % len(channels)],
        "amount_cents": 950 + (i * 37) % 6_000,
        "delivery_note": f"order-{i:05d} / leave at pickup desk",
    }
    for i in range(1, n_orders + 1)
]
orders_table = pa.Table.from_pylist(orders)
shuffled_rows = list(range(n_orders))
Random(42).shuffle(shuffled_rows)
mixed_table = orders_table.take(shuffled_rows)


def write_parquet(table, path, compression="zstd"):
    pq.write_table(table, path, row_group_size=rows_per_group, compression=compression)


sorted_path, mixed_path = root / "by-id.parquet", root / "mixed.parquet"
write_parquet(orders_table, sorted_path)
write_parquet(mixed_table, mixed_path)
show_table(list(orders[0]), (row.values() for row in orders[:4]))

### 3а. Проекция колонки `amount_cents`

Представьте запрос `SELECT SUM(amount_cents) FROM orders`. Он использует одну колонку из шести. Откроем футер, найдём размеры её column chunks и сравним их с размерами всех колонок.

In [ ]:
metadata = pq.read_metadata(sorted_path)
all_groups = [metadata.row_group(i) for i in range(metadata.num_row_groups)]
projection = ["amount_cents"]
column_bytes = {
    name: sum(group.column(i).total_compressed_size for group in all_groups)
    for i, name in enumerate(metadata.schema.names)
}
show_table(
    ["Колонка", "Размер chunks, байт", "Выбрана для чтения"],
    [(name, f"{size:,}", "да" if name in projection else "—")
     for name, size in column_bytes.items()],
)
print(f"Row groups: {metadata.num_row_groups}")
print(f"Все колонки: {sum(column_bytes.values()):,} байт")
print(f"Проекция: {sum(column_bytes[name] for name in projection):,} байт")
projected = pq.read_table(sorted_path, columns=projection)
total = pc.sum(projected["amount_cents"]).as_py()
assert projected.column_names == projection
assert total == sum(row["amount_cents"] for row in orders)
print(f"SUM(amount_cents) = {total:,}")

Размеры в таблице взяты из футера. Фактический ввод-вывод также зависит от метаданных, упреждающего чтения и кеша.

### 3б. Отсечение row groups по `min/max`

**Предскажите результат.** Сколько групп останется при `WHERE order_id BETWEEN 4500 AND 4599` в каждом файле?

Покажем диапазоны из футера и группы, которые PyArrow оставляет для сканирования. Пересечение диапазона с фильтром означает, что группа **может** содержать нужные строки. Сам фильтр ещё применяется к строкам внутри оставленных групп. [Документация PyArrow по отсечению row groups](https://arrow.apache.org/docs/21.0/python/generated/pyarrow.dataset.ParquetFileFragment.html#pyarrow.dataset.ParquetFileFragment.split_by_row_group)

In [ ]:
first_id, last_id = 4_500, 4_599
predicate = (ds.field("order_id") >= first_id) & (ds.field("order_id") <= last_id)
results = []
group_rows = []
for label, path in [("По order_id", sorted_path), ("Перемешано", mixed_path)]:
    dataset = ds.dataset(path, format="parquet")
    fragment = next(dataset.get_fragments())
    kept = {
        info.id
        for piece in fragment.split_by_row_group(filter=predicate)
        for info in piece.row_groups
    }
    for index in range(fragment.metadata.num_row_groups):
        stats = fragment.metadata.row_group(index).column(0).statistics
        group_rows.append((label, index, stats.min, stats.max, "**да**" if index in kept else "—"))

    result = dataset.to_table(columns=["order_id", "amount_cents"], filter=predicate)
    results.append(result.sort_by([("order_id", "ascending")]))
    print(f"{label}: групп-кандидатов {len(kept)} из {fragment.metadata.num_row_groups}")

show_table(["Порядок строк", "Row group", "min(order_id)", "max(order_id)", "Кандидат"], group_rows)
assert results[0].equals(results[1])
assert results[0]["order_id"].to_pylist() == list(range(first_id, last_id + 1))
print(f"В обоих файлах найдено {results[0].num_rows} заказов.")

## 4. Отсечение файлов по партициям

Запишем те же заказы в папки вида `market=eu/`. Hive-разметка позволяет восстановить `market` из пути. Сравним список всех файлов с кандидатами для `WHERE market = 'eu'` и заглянем в схему одного файла.

Здесь движок отбрасывает файлы по пути. В предыдущем опыте он отбрасывал группы внутри файла по статистике в футере.

In [ ]:
partition_path = root / "orders_by_market"
ds.write_dataset(
    orders_table,
    partition_path,
    format="parquet",
    partitioning=["market"],
    partitioning_flavor="hive",
    existing_data_behavior="overwrite_or_ignore",
)
partitioned = ds.dataset(partition_path, format="parquet", partitioning="hive")
market_filter = ds.field("market") == "eu"
all_files = sorted(partitioned.get_fragments(), key=lambda fragment: fragment.path)
eu_files = list(partitioned.get_fragments(filter=market_filter))
eu_paths = {fragment.path for fragment in eu_files}

show_table(
    ["Файл", "Кандидат для market = 'eu'"],
    [(Path(fragment.path).relative_to(partition_path),
      "**да**" if fragment.path in eu_paths else "—")
     for fragment in all_files],
)
print("Колонки в физическом файле:", ", ".join(eu_files[0].physical_schema.names))
print("Колонки датасета:", ", ".join(partitioned.schema.names))

eu_orders = partitioned.to_table(columns=["order_id", "market"], filter=market_filter)
assert "market" not in eu_files[0].physical_schema.names
assert set(eu_orders["market"].to_pylist()) == {"eu"}
assert eu_orders.num_rows == sum(row["market"] == "eu" for row in orders)
print(f"Файлов-кандидатов: {len(eu_files)} из {len(all_files)}. Заказов: {eu_orders.num_rows:,}.")

## 5. Влияние формата, кодека и порядка строк на размер

1. Пишем **одни и те же заказы** в CSV, NDJSON (один JSON-объект на строку), Avro с кодеком `null` и Parquet без кодека. У Parquet остаётся собственное кодирование колонок, в том числе dictionary encoding.
2. Включаем ZSTD для того же Parquet, сохраняя порядок строк, размеры групп и прочие настройки.
3. При том же ZSTD сравниваем только chunks колонки `market` в двух файлах: с перемешанными строками и с сортировкой по `market`. Размер всего файла может меняться иначе, потому что меняется порядок значений и в других колонках.

**Предскажите результат.** Какой текстовый файл будет больше? Что изменится в chunks колонки `market`, если одинаковые значения собрать рядом?

In [ ]:
import csv

csv_path, json_path = root / "orders.csv", root / "orders.ndjson"
avro_path, plain_parquet_path = root / "orders.avro", root / "plain.parquet"

with csv_path.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(orders[0]))
    writer.writeheader()
    writer.writerows(orders)

with json_path.open("w", encoding="utf-8") as file:
    for order in orders:
        file.write(json.dumps(order, separators=(",", ":")) + "\n")

order_schema = avro.schema.parse(json.dumps({
    "type": "record",
    "name": "Order",
    "fields": [
        {"name": "order_id", "type": "long"},
        {"name": "book_id", "type": "string"},
        {"name": "market", "type": "string"},
        {"name": "channel", "type": "string"},
        {"name": "amount_cents", "type": "long"},
        {"name": "delivery_note", "type": "string"},
    ],
}))
write_avro(avro_path, order_schema, orders)
write_parquet(orders_table, plain_parquet_path, compression=None)

sizes = [
    ("CSV", csv_path.stat().st_size),
    ("NDJSON", json_path.stat().st_size),
    ("Avro, null", avro_path.stat().st_size),
    ("Parquet, без кодека", plain_parquet_path.stat().st_size),
    ("Parquet, ZSTD", sorted_path.stat().st_size),
]

In [ ]:
# SVG строится по размерам файлов на диске.
chart = [
    '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 660 236" width="660" role="img">',
    '<title>Размер файлов с 12 000 заказами, КиБ</title>',
    '<rect width="660" height="236" fill="#f8fafc"/>',
    '<g font-family="sans-serif" font-size="14" fill="#17263c">',
    '<text x="12" y="26" font-weight="bold">Размер файлов, КиБ</text>',
]
scale = 355 / max(size for _, size in sizes)
for index, (label, size) in enumerate(sizes):
    y = 44 + 35 * index
    bar_width = size * scale
    color = "#087c73" if label == "Parquet, ZSTD" else "#325ba8"
    chart.extend([
        f'<text x="12" y="{y + 16}">{label}</text>',
        f'<rect x="174" y="{y}" width="{bar_width:.1f}" height="22" rx="3" fill="{color}"/>',
        f'<text x="{182 + bar_width:.1f}" y="{y + 16}">{size / 1024:,.1f}</text>',
    ])
chart.append("</g></svg>")
display(SVG("".join(chart)))

market_sorted_path = root / "by-market.parquet"
write_parquet(orders_table.sort_by([("market", "ascending")]), market_sorted_path)
market_index = orders_table.schema.get_field_index("market")
sorting_rows = []
for label, path in [("Перемешано", mixed_path), ("По market", market_sorted_path)]:
    footer = pq.read_metadata(path)
    market_chunks = [
        footer.row_group(i).column(market_index) for i in range(footer.num_row_groups)
    ]
    sorting_rows.append((
        label,
        ", ".join(market_chunks[0].encodings),
        f"{sum(chunk.total_compressed_size for chunk in market_chunks):,}",
        f"{path.stat().st_size:,}",
    ))

show_table(["Порядок строк", "Кодировки market", "Chunks market, байт", "Весь файл, байт"], sorting_rows)

## Что можно проверить на семинаре

- Поменяйте границы фильтра на `3900–4100`. Сколько групп придётся прочитать в файле, отсортированном по `order_id`?
- Добавьте к проекции `delivery_note`. Как изменится сумма размеров chunks и почему?
- Задайте `default_currency = "JPY"` и перезапустите обе ячейки Avro. У какой комбинации файла и читателя поменяется результат?

В опытах с Parquet работали с файлами, путями и футерами. Отдельное [демо Iceberg](README.md) продолжает лекцию на уровне таблицы, где для планирования чтения используются снапшоты и манифесты.